In [102]:
import pandas as pd
import json
import numpy as np
import openpyxl

In [103]:
all_essays_AES = pd.read_csv("../data/random_essays_50.csv", sep='\t', encoding='utf-8', on_bad_lines='skip')
all_essays_ivypanda = pd.read_csv("../data/ivypanda.csv")

In [104]:
all_data_ivypanda = []
for index, row in all_essays_ivypanda.iterrows():
    essay_id = row['essay_id']
    filepath = f'../output/ivypanda/win/chat_output_' + str(essay_id) + '.txt'
    with open(filepath, 'r', encoding='utf-8') as file:
        # 读入json文件
        data = json.load(file)
        all_data_ivypanda.append(data)

all_data_AES = []
for index, row in all_essays_AES.iterrows():
    essay_id = row['essay_id']
    filepath = f'../output/AES/win/chat_output_' + str(essay_id) + '.txt'
    with open(filepath, 'r', encoding='utf-8') as file:
        # 读入json文件
        data = json.load(file)
        all_data_AES.append(data)  

In [110]:
# 初始化存储统计数据的字典
statistics = {'A': {}, 'B': {}, 'C': {}}

# 遍历所有数据项
for item in all_data_ivypanda:
    reviews = item['reviews']
    for person, categories in reviews.items():
        for category, score in categories.items():
            if category not in statistics[person]:
                statistics[person][category] = []
            statistics[person][category].append(score)

# 定义计算统计数据的函数
def calculate_statistics(scores):
    mean = np.mean(scores)
    median = np.median(scores)
    variance = np.var(scores)
    std_dev = np.std(scores)
    range_val = np.max(scores) - np.min(scores)
    max_val = np.max(scores)
    min_val = np.min(scores)
    return {'Mean': mean, 'Median': median, 'Variance': variance, 'Standard Deviation': std_dev, 'Range': range_val, 'Max': max_val, 'Min': min_val}

# 计算统计数据
for person, categories in statistics.items():
    for category, scores in categories.items():
        statistics[person][category] = calculate_statistics(scores)

# 输出统计数据
for person, stats in statistics.items():
    df = pd.DataFrame(stats).T
    print(f"Statistics for {person}:")
    print(df)
    print("\n")
    df.to_csv(f'../exp_analysis/statistics_{person}.csv', index=True, header=True)


Statistics for A:
                  Mean  Median  Variance  Standard Deviation  Range  Max  Min
Intention         8.06     8.0    0.1364            0.369324    2.0  9.0  7.0
Genre             7.64     7.0    0.6704            0.818780    2.0  9.0  7.0
Contradiction     8.58     9.0    0.2436            0.493559    1.0  9.0  8.0
Specificity       7.22     7.0    0.2916            0.540000    3.0  9.0  6.0
Constructiveness  8.06     8.0    0.3364            0.580000    2.0  9.0  7.0
Depth             6.80     7.0    0.4000            0.632456    2.0  8.0  6.0


Statistics for B:
                  Mean  Median  Variance  Standard Deviation  Range   Max  Min
Intention         7.80     7.0    0.9200            0.959166    2.0   9.0  7.0
Genre             7.72     8.0    0.4016            0.633719    3.0   9.0  6.0
Contradiction     8.32     8.0    0.4176            0.646220    3.0  10.0  7.0
Specificity       7.10     7.0    0.7300            0.854400    2.0   8.0  6.0
Constructiveness  7.8

In [126]:
df_feedback_AES = pd.read_csv("../output/AES/debate/0.feedbacks.csv", sep='\t', encoding='utf-8', on_bad_lines='skip')

df_feedback_ivypanda = pd.read_csv("../output/ivypanda/debate/0.feedbacks.csv", sep='\t', encoding='utf-8', on_bad_lines='skip')

# 定义作者意图分类
intention_categories = [
    'Explain/Clarify', 'Educate/Teach', 'Persuade/Influence', 'Entertain/Engage',
    'Analyze/Critique', 'Describe/Depict', 'Reflect/Self-Express', 'Solve Problems',
    'Preserve/Record', 'Innovate/Experiment'
]

# 初始化计数字典
intention_counts_AES = {category: 0 for category in intention_categories}

# 计算每个意图出现的次数
for category in intention_categories:
    intention_counts_AES[category] = df_feedback_AES[df_feedback_AES['intention'] == category].shape[0]

intention_counts_ivypanda = {category: 0 for category in intention_categories}
# 计算每个意图出现的次数
for category in intention_categories:
    intention_counts_ivypanda[category] = df_feedback_ivypanda[df_feedback_ivypanda['intention'] == category].shape[0]

# add up
intention_counts = {category: intention_counts_AES[category] + intention_counts_ivypanda[category] for category in intention_categories}


display(intention_counts_AES)
display(intention_counts_ivypanda)
display(intention_counts)


{'Explain/Clarify': 0,
 'Educate/Teach': 3,
 'Persuade/Influence': 30,
 'Entertain/Engage': 3,
 'Analyze/Critique': 0,
 'Describe/Depict': 4,
 'Reflect/Self-Express': 9,
 'Solve Problems': 0,
 'Preserve/Record': 0,
 'Innovate/Experiment': 1}

{'Explain/Clarify': 4,
 'Educate/Teach': 23,
 'Persuade/Influence': 3,
 'Entertain/Engage': 0,
 'Analyze/Critique': 20,
 'Describe/Depict': 0,
 'Reflect/Self-Express': 0,
 'Solve Problems': 0,
 'Preserve/Record': 0,
 'Innovate/Experiment': 0}

{'Explain/Clarify': 4,
 'Educate/Teach': 26,
 'Persuade/Influence': 33,
 'Entertain/Engage': 3,
 'Analyze/Critique': 20,
 'Describe/Depict': 4,
 'Reflect/Self-Express': 9,
 'Solve Problems': 0,
 'Preserve/Record': 0,
 'Innovate/Experiment': 1}

In [ ]:
# all_data是all_data_AES和all_data_ivypanda的合并
all_data = all_data_AES + all_data_ivypanda
df_feedback = pd.concat([df_feedback_AES, df_feedback_ivypanda], ignore_index=True)

for i in range(0,100):
    essay_id = df_feedback.iloc[i]['essay_id']
    data = all_data[i]
    data['intention'] = df_feedback.iloc[i]['intention']
    data['essay_id'] = int(essay_id)
    all_data[i] = data


[{'reviews': {'A': {'Intention': 9,
    'Genre': 8,
    'Contradiction': 9,
    'Specificity': 8,
    'Constructiveness': 9,
    'Depth': 8},
   'B': {'Intention': 7,
    'Genre': 7,
    'Contradiction': 8,
    'Specificity': 7,
    'Constructiveness': 7,
    'Depth': 6},
   'C': {'Intention': 8,
    'Genre': 9,
    'Contradiction': 9,
    'Specificity': 9,
    'Constructiveness': 9,
    'Depth': 8}},
  'best_review': 'A',
  'intention': 'Persuade/Influence',
  'essay_id': 24},
 {'reviews': {'A': {'Intention': 8,
    'Genre': 7,
    'Contradiction': 9,
    'Specificity': 8,
    'Constructiveness': 9,
    'Depth': 7},
   'B': {'Intention': 7,
    'Genre': 7,
    'Contradiction': 8,
    'Specificity': 7,
    'Constructiveness': 8,
    'Depth': 6},
   'C': {'Intention': 9,
    'Genre': 8,
    'Contradiction': 9,
    'Specificity': 9,
    'Constructiveness': 9,
    'Depth': 8}},
  'best_review': 'C',
  'intention': 'Persuade/Influence',
  'essay_id': 200},
 {'reviews': {'A': {'Intention': 

In [108]:
df_all = pd.json_normalize(all_data)
df_all

,best_review,intention,essay_id,reviews.A.Intention,reviews.A.Genre,reviews.A.Contradiction,reviews.A.Specificity,reviews.A.Constructiveness,reviews.A.Depth,reviews.B.Intention,...,reviews.B.Contradiction,reviews.B.Specificity,reviews.B.Constructiveness,reviews.B.Depth,reviews.C.Intention,reviews.C.Genre,reviews.C.Contradiction,reviews.C.Specificity,reviews.C.Constructiveness,reviews.C.Depth
0,A,Persuade/Influence,24,9,8,9,8,9,8,7,...,8,7,7,6,8,9,9,9,9,8
1,C,Persuade/Influence,200,8,7,9,8,9,7,7,...,8,7,8,6,9,8,9,9,9,8
2,C,Persuade/Influence,222,9,8,9,8,9,8,7,...,8,7,7,7,8,9,9,9,9,9
3,A,Persuade/Influence,411,9,8,9,8,9,8,7,...,8,7,7,7,8,9,9,9,9,8
4,C,Persuade/Influence,657,8,7,8,7,8,6,9,...,9,8,9,7,10,9,10,9,10,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,C,Analyze/Critique,45,8,8,9,7,8,7,7,...,8,6,7,6,9,9,10,9,9,8
96,C,Analyze/Critique,46,8,9,9,7,8,7,7,...,8,6,7,6,9,10,10,9,9,8
97,C,Analyze/Critique,47,8,7,8,7,8,7,9,...,9,8,9,8,10,9,10,9,10,9
98,C,Educate/Teach,48,8,7,9,7,7,6,9,...,8,8,8,7,10,9,10,9,9,8


In [109]:
columns = df_all.columns
columns = columns.drop(['intention', 'essay_id', 'best_review'])
# print(columns)
df_score = df_all.groupby('intention').agg(
    {
        # columns列表中所有的列都计算mean
        **{col: 'mean' for col in columns},
    }
)
df_score.to_csv('../exp_analysis/groupby_intention_score.csv', sep='\t', encoding='utf-8', index=True)

In [124]:
df_score = pd.read_csv('groupby_intention_score.csv', sep='\t', encoding='utf-8', on_bad_lines='skip')
df_cmp = pd.DataFrame()

df_cmp['C2A_intention'] = df_score['reviews.A.Intention'] - df_score['reviews.C.Intention']
df_cmp['C2A_genre'] = df_score['reviews.A.Genre'] - df_score['reviews.C.Genre']
df_cmp['C2A_contradiction'] = df_score['reviews.A.Contradiction'] - df_score['reviews.C.Contradiction']
df_cmp['C2A_specificity'] = df_score['reviews.A.Specificity'] - df_score['reviews.C.Specificity']
df_cmp['C2A_constructiveness'] = df_score['reviews.A.Constructiveness'] - df_score['reviews.C.Constructiveness']
df_cmp['C2A_depth'] = df_score['reviews.A.Depth'] - df_score['reviews.C.Depth']
df_cmp['C2B_intention'] = df_score['reviews.B.Intention'] - df_score['reviews.C.Intention']
df_cmp['C2B_genre'] = df_score['reviews.B.Genre'] - df_score['reviews.C.Genre']
df_cmp['C2B_contradiction'] = df_score['reviews.B.Contradiction'] - df_score['reviews.C.Contradiction']
df_cmp['C2B_specificity'] = df_score['reviews.B.Specificity'] - df_score['reviews.C.Specificity']
df_cmp['C2B_constructiveness'] = df_score['reviews.B.Constructiveness'] - df_score['reviews.C.Constructiveness']
df_cmp['C2B_depth'] = df_score['reviews.B.Depth'] - df_score['reviews.C.Depth']
df_cmp = -df_cmp
# 在df_cmp最前面添加intention列
df_cmp.insert(0, 'intention', df_score['intention'])

df_cmp.to_csv('groupby_intention_score_cmp.csv', index=False)